# Policy composition, activation weight, and normalized payoff

Rows show population sizes and columns show agent groups. The horizontal axis is the realized payoff-guided share, the vertical axis is each random agent's activation weight relative to a unit-weight payoff-guided agent, and color is expected payoff at the common normalized-time endpoint. Composition is sampled independently and evenly over all realizable agent counts; the remaining parameters use the configured space-filling sample.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib_inline.backend_inline import set_matplotlib_formats
import numpy as np
import scienceplots  # noqa: F401
from statsmodels.nonparametric.smoothers_lowess import lowess
import xarray as xr

plt.style.use("science")
set_matplotlib_formats("svg")
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "figure.titlesize": 20,
})

project_root = Path.cwd().resolve()
if not (project_root / "tasks").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tasks.policies import load_policy_results

In [ ]:
config_path = project_root / "configs/best_response_random.baseline.run_config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
expected_policy_config = {
    key: config[key]
    for key in ("sampling", "simulator", "game", "population", "observers")
}

def matches_config(path):
    try:
        attrs = json.loads((path / ".zattrs").read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return False
    return (
        attrs.get("sweep_layout") == "sampled_composition"
        and attrs.get("policy_config") == expected_policy_config
    )

run_dir = project_root / "artifacts/simulation_runs"
matching_runs = sorted(path for path in run_dir.glob("run-*.zarr") if matches_config(path))
if not matching_runs:
    raise FileNotFoundError(
        f"No completed run matches {config_path.relative_to(project_root)}"
    )
result_store_path = matching_runs[-1]
ds = load_policy_results(
    result_store_path,
    stats=["policy_welfare.mean"],
    normalize_time=True,
)
if ds.attrs["has_failures"]:
    raise RuntimeError(f"Run contains failed rows: {result_store_path}")
print(
    f"Loaded {result_store_path.relative_to(project_root)} at "
    f"{float(ds.step[-1]):g} activation opportunities per agent"
)

In [ ]:
policy_payoff = ds["policy_welfare.mean"].isel(step=-1, drop=True)
best_response_payoff = policy_payoff.sel(policy_class="best_response", drop=True)
random_payoff = policy_payoff.sel(policy_class="random", drop=True)
best_response_count = ds.coords["best_response_count"]
random_count = ds.coords["random_count"]
n_agents = ds.coords["n_agents"]
everyone_payoff = (
    xr.where(best_response_count > 0, best_response_count * best_response_payoff, 0.0)
    + xr.where(random_count > 0, random_count * random_payoff, 0.0)
) / n_agents

population_groups = ("All agents", "Payoff-guided agents", "Random-action agents")
group_payoff = xr.concat(
    [everyone_payoff, best_response_payoff, random_payoff],
    dim=xr.IndexVariable("population_group", population_groups),
).rename("mean_normalized_payoff")
random_activation_weight = ds["theta"].sel(
    theta_parameter="random.agent.activation_weight",
    drop=True,
).rename("random_activation_weight")
plot_frame = xr.Dataset({
    "mean_normalized_payoff": group_payoff,
    "random_activation_weight": random_activation_weight,
}).to_dataframe().reset_index()
plot_frame = plot_frame.dropna(
    subset=["mean_normalized_payoff", "random_activation_weight"]
)

coverage = (
    plot_frame[plot_frame["population_group"] == "All agents"]
    .groupby(["n_agents", "best_response_proportion"])
    .size()
)
print(f"Samples per realized composition: {coverage.min()} to {coverage.max()}")

In [ ]:
def centers_to_edges(values, *, lower=None, upper=None):
    values = np.asarray(values, dtype=float)
    midpoints = (values[:-1] + values[1:]) / 2
    edges = np.concatenate((
        [values[0] - (midpoints[0] - values[0])],
        midpoints,
        [values[-1] + (values[-1] - midpoints[-1])],
    ))
    if lower is not None:
        edges[0] = lower
    if upper is not None:
        edges[-1] = upper
    return edges


def payoff_surface(panel, proportions, weight_grid, *, lowess_frac=0.35):
    surface = np.full((weight_grid.size, proportions.size), np.nan)
    for column, proportion in enumerate(proportions):
        series = panel[np.isclose(panel["best_response_proportion"], proportion)]
        if series.empty:
            continue
        x = series["random_activation_weight"].to_numpy(dtype=float)
        y = series["mean_normalized_payoff"].to_numpy(dtype=float)
        if proportion in (0.0, 1.0):
            surface[:, column] = np.mean(y)
            continue
        order = np.argsort(x)
        fitted = lowess(
            y[order],
            x[order],
            frac=lowess_frac,
            it=0,
            is_sorted=True,
            return_sorted=True,
        )
        surface[:, column] = np.interp(
            weight_grid, fitted[:, 0], fitted[:, 1]
        )
    return surface


def build_surfaces(frame, *, weight_points=160, lowess_frac=0.35):
    weight_grid = np.linspace(
        frame["random_activation_weight"].min(),
        frame["random_activation_weight"].max(),
        weight_points,
    )
    surfaces = {}
    for n_agents_value in np.sort(frame["n_agents"].unique()):
        population = frame[frame["n_agents"] == n_agents_value]
        proportions = np.sort(population["best_response_proportion"].unique())
        for group in population_groups:
            panel = population[population["population_group"] == group]
            surface = payoff_surface(
                panel, proportions, weight_grid, lowess_frac=lowess_frac
            )
            if group == "Payoff-guided agents":
                surface[:, np.isclose(proportions, 0.0)] = np.nan
            elif group == "Random-action agents":
                surface[:, np.isclose(proportions, 1.0)] = np.nan
            surfaces[(n_agents_value, group)] = (proportions, surface)
    return weight_grid, surfaces

In [ ]:
weight_grid, surfaces = build_surfaces(plot_frame)
finite_values = np.concatenate([
    surface[np.isfinite(surface)] for _, surface in surfaces.values()
])
vmin = min(float(np.percentile(finite_values, 1)), -0.01)
vmax = max(float(np.percentile(finite_values, 99)), 0.01)
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
agent_counts = np.sort(plot_frame["n_agents"].unique())
fig, axes = plt.subplots(
    len(agent_counts),
    len(population_groups),
    figsize=(15, 9.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
    squeeze=False,
)
weight_edges = centers_to_edges(weight_grid)
mesh = None
for row, n_agents_value in enumerate(agent_counts):
    for column, group in enumerate(population_groups):
        ax = axes[row, column]
        proportions, surface = surfaces[(n_agents_value, group)]
        proportion_edges = centers_to_edges(proportions, lower=0.0, upper=1.0)
        mesh = ax.pcolormesh(
            proportion_edges,
            weight_edges,
            surface,
            shading="flat",
            cmap="RdBu_r",
            norm=norm,
            rasterized=True,
        )
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(weight_grid[0], weight_grid[-1])
        ax.grid(color="white", alpha=0.18, linewidth=0.6)
        if row == 0:
            ax.set_title(group, fontweight="bold")
        if column == 0:
            ax.set_ylabel(f"{int(n_agents_value)} agents\nRandom-agent weight")
        if row == len(agent_counts) - 1:
            ax.set_xlabel("Payoff-guided proportion")

assert mesh is not None
colorbar = fig.colorbar(mesh, ax=axes, location="right", shrink=0.78, pad=0.02)
colorbar.set_label("Mean normalized payoff")
fig.suptitle(
    f"Payoff at {float(ds.step[-1]):g} activation opportunities per agent"
)
plt.show()

In [ ]:
run_id = result_store_path.stem
figure_output_dir = project_root / "artifacts/figures" / run_id
figure_output_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_output_dir / "best_response_proportion_effect.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")